# Exploration: RTE `total_forecast`

Notebook d'exploration pour:
- générer un token OAuth RTE depuis un `Basic` base64 éditable,
- appeler l'endpoint `generation_forecast/v3/total_forecast`,
- sur la période **de deux jours autours de la date actuelle** en **heure française** (`Europe/Paris`).


In [72]:
from __future__ import annotations

import datetime as dt
from zoneinfo import ZoneInfo

import httpx


In [73]:
# A MODIFIER: colle ici la valeur base64 de "client_id:client_secret"
RTE_BASIC_AUTH_B64 = "MTg5OTg3ZmYtMmQzYS00YmYzLWFjNGMtMTA0OGZkMzk3YWU4OjBlMmQyY2UzLTU2Y2EtNGNmZS1hMTlhLWYwZDg0NzcxOGU4Zg=="

TOKEN_URL = "https://digital.iservices.rte-france.com/token/oauth/"
TOTAL_FORECAST_URL = "https://digital.iservices.rte-france.com/open_api/generation_forecast/v3/total_forecast"
PARIS_TZ = ZoneInfo("Europe/Paris")

# Optionnel: types de forecast à demander (laisser [] pour ne pas envoyer le paramètre)
TOTAL_FORECAST_TYPES: list[str] = ["D-1", "ID"]

today = dt.datetime.now(PARIS_TZ)

start_date_paris = today - dt.timedelta(days=2)
end_date_paris = today + dt.timedelta(days=2)

# does not work yet, I don't know why
# today = dt.datetime.now(PARIS_TZ).date()
# start_date_paris = dt.datetime.combine(start_date_paris, dt.time(18,0,0))
# end_date_paris = dt.datetime.combine(end_date_paris, dt.time(16,59,59))

print("today:", today.isoformat())
print("start_date_paris:", start_date_paris.isoformat())
print("end_date_paris:  ", end_date_paris.isoformat())


today: 2026-04-08T17:27:49.371909+02:00
start_date_paris: 2026-04-06T17:27:49.371909+02:00
end_date_paris:   2026-04-10T17:27:49.371909+02:00


In [74]:
if "PASTE_BASE64" in RTE_BASIC_AUTH_B64:
    raise ValueError("Renseigne d'abord RTE_BASIC_AUTH_B64 avec ta vraie valeur base64.")

with httpx.Client(timeout=30.0) as client:
    token_response = client.post(
        TOKEN_URL,
        headers={
            "Authorization": f"Basic {RTE_BASIC_AUTH_B64}",
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
        },
        data={},
    )

token_response.raise_for_status()
token_payload = token_response.json()
access_token = token_payload.get("access_token")

if not access_token:
    raise RuntimeError(f"Token RTE absent de la reponse: {token_payload}")

print("Token recupere (tronque):", access_token[:12] + "...")


Token recupere (tronque): 500f1WdlImpn...


In [75]:
access_token

'500f1WdlImpnYrzS5Xpd2c0vRugym2r6P1Zi4o10BudxPgpx5phuZD'

In [76]:
params = {
    "start_date": start_date_paris.isoformat(),
    "end_date": end_date_paris.isoformat(),
}
if TOTAL_FORECAST_TYPES:
    params["type"] = ",".join(dict.fromkeys(TOTAL_FORECAST_TYPES))

with httpx.Client(timeout=60.0) as client:
    total_forecast_response = client.get(
        TOTAL_FORECAST_URL,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Accept": "application/json",
        },
        params=params,
    )

total_forecast_response.raise_for_status()
total_forecast_payload = total_forecast_response.json()

print("Status:", total_forecast_response.status_code)
print("Keys:", list(total_forecast_payload.keys()))


Status: 200
Keys: ['total_forecast']


In [78]:
import sys
sys.getsizeof(total_forecast_payload) # returns it in bytes.

184

In [79]:
total_forecast_payload

{'total_forecast': [{'points': [{'quantity': 49821,
     'start_date': '2026-04-06T15:30:00Z'},
    {'quantity': 49832, 'start_date': '2026-04-06T15:45:00Z'},
    {'quantity': 51626, 'start_date': '2026-04-06T16:00:00Z'},
    {'quantity': 50974, 'start_date': '2026-04-06T16:15:00Z'},
    {'quantity': 52532, 'start_date': '2026-04-06T16:30:00Z'},
    {'quantity': 54303, 'start_date': '2026-04-06T16:45:00Z'},
    {'quantity': 56064, 'start_date': '2026-04-06T17:00:00Z'},
    {'quantity': 56734, 'start_date': '2026-04-06T17:15:00Z'},
    {'quantity': 57983, 'start_date': '2026-04-06T17:30:00Z'},
    {'quantity': 57904, 'start_date': '2026-04-06T17:45:00Z'},
    {'quantity': 58521, 'start_date': '2026-04-06T18:00:00Z'},
    {'quantity': 58485, 'start_date': '2026-04-06T18:15:00Z'},
    {'quantity': 58965, 'start_date': '2026-04-06T18:30:00Z'},
    {'quantity': 59517, 'start_date': '2026-04-06T18:45:00Z'},
    {'quantity': 59350, 'start_date': '2026-04-06T19:00:00Z'},
    {'quantity': 59568

In [80]:
# Apercu rapide des donnees retournees
series = total_forecast_payload["total_forecast"]
print("Nombre de series:", len(series))

if series:
    first_series = series[0]["points"]
    # print("Type:", first_series.get("type"))
    # values = first_series.get("values", [])
    print("Nombre de points:", len(first_series))
    print("Premier point:", first_series[0] if first_series else None)


Nombre de series: 1
Nombre de points: 383
Premier point: {'quantity': 49821, 'start_date': '2026-04-06T15:30:00Z'}


In [81]:
today.strftime("%Y-%m-%d")

tomorrow = today + dt.timedelta(days=1)
tomorrow.strftime("%Y-%m-%d")

'2026-04-09'

In [82]:
# Montrer valeurs pour le jour en cours
tomorrow = today + dt.timedelta(days=1)

for dico in total_forecast_payload["total_forecast"][0]["points"]:
    if (dico["start_date"] >= today.strftime("%Y-%m-%d")) & (dico["start_date"] < tomorrow.strftime("%Y-%m-%d")):
        print(dico["start_date"])
        print(dico["quantity"])

# attention, c'est une erreur ici de ne pas traduire en temps Europe/Paris avant de filter

2026-04-08T00:00:00Z
57325
2026-04-08T00:15:00Z
56984
2026-04-08T00:30:00Z
56289
2026-04-08T00:45:00Z
56075
2026-04-08T01:00:00Z
56120
2026-04-08T01:15:00Z
55509
2026-04-08T01:30:00Z
55196
2026-04-08T01:45:00Z
55101
2026-04-08T02:00:00Z
55760
2026-04-08T02:15:00Z
55718
2026-04-08T02:30:00Z
55904
2026-04-08T02:45:00Z
55937
2026-04-08T03:00:00Z
56482
2026-04-08T03:15:00Z
56838
2026-04-08T03:30:00Z
57337
2026-04-08T03:45:00Z
57990
2026-04-08T04:00:00Z
58990
2026-04-08T04:15:00Z
59686
2026-04-08T04:30:00Z
60906
2026-04-08T04:45:00Z
60924
2026-04-08T05:00:00Z
61439
2026-04-08T05:15:00Z
61818
2026-04-08T05:30:00Z
62071
2026-04-08T05:45:00Z
62701
2026-04-08T06:00:00Z
62871
2026-04-08T06:15:00Z
63404
2026-04-08T06:30:00Z
64171
2026-04-08T06:45:00Z
65107
2026-04-08T07:00:00Z
64334
2026-04-08T07:15:00Z
64957
2026-04-08T07:30:00Z
65964
2026-04-08T07:45:00Z
62956
2026-04-08T08:00:00Z
64554
2026-04-08T08:15:00Z
64567
2026-04-08T08:30:00Z
63964
2026-04-08T08:45:00Z
59605
2026-04-08T09:00:00Z
60009
2

In [83]:
# Extra: lire la timezone de dico["start_date"] puis convertir vers PARIS_TZ

raw_start_time = dico["start_date"]
parsed_start_time = dt.datetime.fromisoformat(raw_start_time)

source_tz = parsed_start_time.tzinfo
paris_time = parsed_start_time.astimezone(PARIS_TZ)

print("Timezone source:", source_tz)
print("Date convertie Paris:", paris_time.isoformat())
print(dico["start_date"])

Timezone source: UTC
Date convertie Paris: 2026-04-10T17:00:00+02:00
2026-04-10T15:00:00Z


CCL : 
Je lui ai demandé à partir de 2026-03-24T00:00:00+01:00
il m'a renvoyé à partir de 2026-03-23T23:00:00Z, soit 2026-03-23T00:00:00+01:00
C'est normal.
c'est juste qu'il faut convertir à nouveau en temps europe/paris avant de faire des tris


Fait aussi l'appel aux autres API, pour avoir le poids des réponses:


Short term consumption

In [85]:

start_date_paris = dt.datetime(2026, 3, 24, 0, 0, 0, tzinfo=PARIS_TZ)
end_date_paris = dt.datetime(2026, 3, 28, 23, 59, 59, tzinfo=PARIS_TZ)


params = {
    "start_date": start_date_paris.isoformat(),
    "end_date": end_date_paris.isoformat(),
}


SHORT_TERM_CONSUMPTION_FORECAST_URL = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

# if TOTAL_FORECAST_TYPES:
#     params["type"] = ",".join(dict.fromkeys(TOTAL_FORECAST_TYPES))

with httpx.Client(timeout=60.0) as client:
    short_term_response = client.get(
        SHORT_TERM_CONSUMPTION_FORECAST_URL,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Accept": "application/json",
        },
        params=params,
    )

short_term_response.raise_for_status()
short_term_payload = short_term_response.json()

print("Status:", short_term_response.status_code)
print("Keys:", list(short_term_payload.keys()))


Status: 200
Keys: ['short_term']


In [86]:
sys.getsizeof(short_term_payload)

184

In [87]:
short_term_payload

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-03-24T00:00:00+01:00',
   'end_date': '2026-03-28T00:00:00+01:00',
   'values': [{'start_date': '2026-03-24T00:00:00+01:00',
     'end_date': '2026-03-24T00:15:00+01:00',
     'updated_date': '2026-03-24T23:48:13+01:00',
     'value': 51659},
    {'start_date': '2026-03-24T00:15:00+01:00',
     'end_date': '2026-03-24T00:30:00+01:00',
     'updated_date': '2026-03-24T23:48:13+01:00',
     'value': 51667},
    {'start_date': '2026-03-24T00:30:00+01:00',
     'end_date': '2026-03-24T00:45:00+01:00',
     'updated_date': '2026-03-24T23:48:13+01:00',
     'value': 50893},
    {'start_date': '2026-03-24T00:45:00+01:00',
     'end_date': '2026-03-24T01:00:00+01:00',
     'updated_date': '2026-03-24T23:48:14+01:00',
     'value': 49981},
    {'start_date': '2026-03-24T01:00:00+01:00',
     'end_date': '2026-03-24T01:15:00+01:00',
     'updated_date': '2026-03-24T23:48:14+01:00',
     'value': 49082},
    {'start_date': '2026-03-24T01

PRE

In [106]:
start_date_paris = dt.datetime(2026, 3, 24, 0, 0, 0, tzinfo=PARIS_TZ)
end_date_paris = dt.datetime(2026, 3, 28, 23, 59, 59, tzinfo=PARIS_TZ)


params = {
    "start_date": start_date_paris.isoformat(),
    "end_date": end_date_paris.isoformat(),
}


BALANCING_DATA_URL = "https://digital.iservices.rte-france.com/open_api/balancing_energy/v5/imbalance_data"

# if TOTAL_FORECAST_TYPES:
#     params["type"] = ",".join(dict.fromkeys(TOTAL_FORECAST_TYPES))

with httpx.Client(timeout=60.0) as client:
    imbalance_data_response = client.get(
        BALANCING_DATA_URL,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Accept": "application/json",
        },
        params=params,
    )

imbalance_data_response.raise_for_status()
imbalance_data_payload = imbalance_data_response.json()

print("Status:", imbalance_data_response.status_code)
print("Keys:", list(imbalance_data_payload.keys()))


Status: 200
Keys: ['imbalance_data']


In [ ]:
# # pour mesurer le poids de ces données
# import csv
# with open('imbalance_data_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in imbalance_data_payload.items():
#        writer.writerow([key, value])

# with open('short_term_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in short_term_payload.items():
#        writer.writerow([key, value])

# with open('total_forecast_payload.csv', 'w') as csv_file:  
#     writer = csv.writer(csv_file)
#     for key, value in total_forecast_payload.items():
#        writer.writerow([key, value])

In [108]:
imbalance_data_payload

{'imbalance_data': [{'start_date': '2026-03-24T00:00:00+01:00',
   'end_date': '2026-03-28T23:45:00+01:00',
   'resolution': 'PT15M',
   'values': [{'start_date': '2026-03-28T23:30:00+01:00',
     'end_date': '2026-03-28T23:45:00+01:00',
     'imbalance': 380.92,
     'system_trend': 'BAISSE',
     'positive_imbalance_settlement_price': 41.75,
     'negative_imbalance_settlement_price': 51.77,
     'missing_data_list': 'N/A',
     'updated_date': '2026-03-28T23:47:48+01:00'},
    {'start_date': '2026-03-28T23:15:00+01:00',
     'end_date': '2026-03-28T23:30:00+01:00',
     'imbalance': 458.65,
     'system_trend': 'BAISSE',
     'positive_imbalance_settlement_price': -37.98,
     'negative_imbalance_settlement_price': -30.62,
     'missing_data_list': 'N/A',
     'updated_date': '2026-03-29T08:28:25+02:00'},
    {'start_date': '2026-03-28T23:00:00+01:00',
     'end_date': '2026-03-28T23:15:00+01:00',
     'imbalance': 330.45,
     'system_trend': 'BAISSE',
     'positive_imbalance_sett